# Scripting languages: practical session 6
## Text processing, errors and exceptions

Wednesday, 4 November 2026

In this practical session, we'll look at text processing and exploratory text analysis, we'll bundle our text processing in a class, and we'll handle errors with `try` and `except`. Some material presented in this notebook is based on previous tutorials by Folgert Karsdorp and Maarten Van Gompel.

## Getting the data

We continue our exploration of *the Stories of One Thousand and One Nights* (practical session 4), and we'll also use an excerpt from the English Wikipedia. The two commands below download and unpack the data.

**A note for Windows users.** The commands with `!` below (`wget`, `unzip`) are unix commands. They work in Google Colab, whatever computer you use. If you run this notebook locally in Jupyter on Windows, download the archive by hand by opening [http://ccl.kuleuven.be/Courses/scripting/scripting_practical6_data.zip](http://ccl.kuleuven.be/Courses/scripting/scripting_practical6_data.zip) in your browser, and unzip it in the folder where this notebook runs.

In [ ]:
!wget http://ccl.kuleuven.be/Courses/scripting/scripting_practical6_data.zip

In [ ]:
!unzip -o -q scripting_practical6_data.zip

## Warming up

Three short warm-up exercises reviewing the previous sessions. They should only take a few minutes each; if not, have another look at practical sessions 4 and 5.

#### Warm-up 1 (paths, practical 4)

In practical session 4, we wrote a function `get_night(filepath)` that extracts the number of a night from its file path: `get_night('arabian_nights/111.txt')` returns the integer `111`. Write it again, using `basename` and `splitext` from `os.path`. We'll need it below to sort the nights.

In [ ]:
from os.path import basename, splitext

def get_night(filepath):
    # insert your code here
    number = basename(filepath)
    split_number = splitext(number)[0]
    return(int(split_number))

# these tests should return True if your code is correct
print(get_night('arabian_nights/111.txt') == 111)
print(get_night('arabian_nights/7.txt') == 7)

#### Warm-up 2 (nested lists, practical 2)

The variable `story` below is a list of sentences, and each sentence is a list of words. Write a function `count_words(sentences)` that returns the total number of words in such a list of lists. Use a `for` loop inside a `for` loop, or `len` on every sentence.

In [ ]:
story = [['this', 'is', 'a', 'story', 'about', 'a', 'cat'], ['the', 'cat', 'is', 'on', 'the', 'mat']]

def count_words(sentences):
    # insert your code here
    count = 0
    for sentence in sentences:
      words_number = len(sentence)
      count += words_number

    return(count)


# this test should return True if your code is correct
print(count_words(story) == 13)

#### Warm-up 3 (classes, practical 5)

Write a class `Story` with a constructor that takes a `title` and a list of sentences (as in warm-up 2), and stores both as attributes. Add a method `word_count()` that returns the number of words (reuse `count_words`). Create a `Story` for the cat story and call the method.

In [ ]:
# insert your code here
class Story:
  def __init__(self,title,sentences):
    self.title = title
    self.sentences = sentences

  def word_count(self):
      count = count_words(self.sentences)
      return(count)

cat = Story('cat_story', story)
cat.word_count()

## Tokenization

Let's first make sure that our stories are properly tokenized. Up till now, we've mostly split our string on whitespace characters in order to detect word boundaries. During the lecture, we discussed why this is suboptimal. Moreover, splitting on whitespace characters does not tell us anything about sentence boundaries. We will therefore make use of the [NLTK](https://www.nltk.org/) library, which includes a simple tokenizer and sentence splitter for English. First, we will import the library, and we'll also download the model for sentence tokenization using the code below:

In [ ]:
import nltk
nltk.download('punkt_tab')

If you're using Colab, `NLTK` should be readily available. If you're using Jupyter notebook on your own computer, you need to make sure to install the module (e.g. using Anaconda Navigator).

We will use the following code to do lower casing, punctuation elimination, tokenization, and sentence splitting with NLTK:

In [ ]:
def tokenize_and_filter(sent):
    return [token.lower() for token in nltk.word_tokenize(sent)
           if token not in ".,?!:;()[]''``*"]

def preprocess(text):
    return [tokenize_and_filter(sent) for sent in nltk.sent_tokenize(text)]

The `tokenize_and_filter` function will process a single sentence. `nltk.word_tokenize` splits words and punctuation into separate tokens; for each resulting token, the lower-case version of tokens is used, and punctuation is filtered out.

The `preprocess` function takes a text as string and does all the processing on it. It first splits the text into a list of sentences with `nltk.sent_tokenize`. It returns the result as a list of sentences, which in turn are lists of tokens, tokenized and cleaned by `tokenize_and_filter`.

In [ ]:
# Let's test the code:
preprocess('This is a sentence. Should we separate it from this one?')

In [ ]:
# What about tricky cases? (This is a single string spread over several lines)
text = """She asked: "Is Dr. Phil a doctor?". "Dr. Phil has a
Ph.D. but is not an M.D." I replied. "He even lost
his license to practice therapy," I added."""

preprocess(text)

Note that for readability, your notebook will print long lists with one element per line.

We can now read our corpus. We collect the file names, sort them by night (with `get_night` from warm-up 1 as the key function), read each file, apply the preprocessing function to it, and collect the result in one big `corpus` list (this may take a minute!):

In [ ]:
from glob import glob

def read_file(filename):
    with open(filename, encoding='utf8') as in_file:
        contents = in_file.read()
    return contents

corpus = []
filenames = glob('arabian_nights/*.txt')
filenames = sorted(filenames, key=get_night)
for filename in filenames:
    text = read_file(filename)
    corpus.append(preprocess(text))

The `preprocess` function already returned two levels of lists (and strings), and now we added yet another level. Do you understand the meaning of each level? Try the following:

In [ ]:
corpus[0]

In [ ]:
corpus[0][0]

In [ ]:
corpus[0][0][0]

In [ ]:
corpus[0][0][0][0]

What does each of the levels in our nested list represent?

## Exploratory data analysis

As a first exploratory data analysis, we are going to compute for each night how many sentences it contains and how many words. It is quite easy to count the number of sentences per night, since each night is represented by a list of sentences.

In [ ]:
sentences_per_night = []
for night in corpus:
    sentences_per_night.append(len(night))
print(sentences_per_night[:10])

Using the function `max` we can find out what the highest number of sentences is:

In [ ]:
max(sentences_per_night)

Similarly, if we would like to know what the lowest number of sentences is, we use the function `min`:

In [ ]:
min(sentences_per_night)

### Exercise 1

By now, you've probably already figured out that the function `sum` takes a list of numbers as input, and returns the sum:

In [ ]:
sum([1, 3, 3, 4])

**(1)** Use this function to compute the average number of sentences per night.

In [ ]:
# insert your code here
total_sentences = sum(sentences_per_night)
total_nights = len(corpus)

average = total_sentences/total_nights

print(average)

**(2)** Given our data structure of a list of sentences which are themselves lists of words, it is a little trickier to count for each night how many words it contains. Create a variable `words_per_night`, similar to the variable `sentences_per_night`, which contains the number of words for each night. Your `count_words` function from warm-up 2 does exactly this for one night.

In [ ]:
# insert your code here
words_per_night = []

for night in corpus:
  words_per_night.append(count_words(night))

print(words_per_night[:10])

The suspense created by Shahrazad's story-telling skills is intriguing, especially the "cliff-hanger" ending each night which she uses to avert her own execution. Every night she tells the Sultan a story only to stop at dawn and she picks up the thread the next night. But does it really take the whole night to tell a particular story?

An average speaker speaks about 100 to 130 words per minute. Narrators are advised to use approximately 150 words per minute in audiobooks. Let's say that for live storytelling, it lies around 120 words per minute (including pauses). Using this information, we can compute the time it takes (in minutes) to tell a particular story as follows:

$$\textrm{story time} = \frac{\textrm{number of words}}{120}$$

### Exercise 2

**(1)** Write a function called `story_time` that takes as input a text that has been tokenized and split into sentences (such as the nights in our corpus). Given a speed of 120 words per minute, compute how long it takes to recite that text.

In [ ]:
def story_time(text):
  # insert yout code here
   total_words = count_words(text)
   time = total_words/120
   return(time)

#these tests should return True if your code is correct
print(story_time(story) == 13 / 120)
print(story_time(corpus[0]) == 11372 / 120)

**(2)** Compute the story time (in minutes) for each night in our corpus. Assign the result to the variable `story_time_per_night`.

In [ ]:
# insert your code here
story_time_per_night = []

for words in words_per_night:
  mins_per_story = words / 120
  story_time_per_night.append(mins_per_story)

print(story_time_per_night[:10])

**(3)** Compute the average, minimum and maximum story telling time.

In [ ]:
# insert your code here

average = sum(story_time_per_night) / len(corpus)

print(average)
print(min(story_time_per_night))
print(max(story_time_per_night))

### Visualizing general statistics

Now that we have computed a range of general statistics for our corpus, it would be nice to visualize them. Python's plotting library *matplotlib* (see [here](http://matplotlib.org)) allows us to produce all kinds of graphs. We import it as follows:

In [ ]:
import matplotlib.pyplot as plt

Now we can, for example, plot for each story how many sentences it contains:

In [ ]:
plt.plot(sentences_per_night)

### Exercise 3

**(1)** Can you do the same for `words_per_night`?

In [ ]:
# insert your code here
plt.plot(words_per_night)

**(2)** And can you do the same for `story_time_per_night`?

In [ ]:
# insert your code here
plt.plot(story_time_per_night)

**(3)** We now want to write a function `positions_of` that returns, for a given word, all sentence numbers in the *Arabian Nights* where that word occurs. We are not interested in the sentence number relative to a particular night, so keep track of sentence numbers for the whole corpus. Use that function to find all occurrences of the name *Shahrazad* and store the corresponding indices in the variable `positions_of_shahrazad`. Do the same thing for the name *Ali*. Store the result in `positions_of_ali`. Finally, find all occurrences of *Egypt* and store the indices in `positions_of_egypt`. Hint: to compute the position in the whole corpus, you will need to keep track of a running total of sentences, using a counter variable that you increase for every sentence. Remember that our corpus is lower-cased.

In [ ]:
# insert your code here
def positions_of(word,text):
  count_sentences = 0
  positions_of = []

  for night in text:
    for sentence in night:
      if word in sentence:
        positions_of.append(count_sentences)
      count_sentences += 1

  return(positions_of)

positions_of_shahrazad = positions_of("shahrazad", corpus)
positions_of_ali = positions_of("ali", corpus)
positions_of_egypt = positions_of("egypt", corpus)


If everything went well, the following lines of code should produce a [dispersion plot](https://medium.com/the-political-ear/tutorial-plotting-lexical-dispersion-conspiracy-lies-from-the-left-of-center-c0b39de442d5) of all sentence occurrences of Shahrazad, Ali and Egypt in the corpus.

In [ ]:
plt.figure(figsize=(20, 8))
names = ['Shahrazad', 'Ali', 'Egypt']
plt.plot(positions_of_egypt, [2] * len(positions_of_egypt), '|', markersize=50)
plt.plot(positions_of_ali, [1] * len(positions_of_ali), '|', markersize=50)
plt.plot(positions_of_shahrazad, [0] * len(positions_of_shahrazad), '|', markersize=50)
plt.yticks(range(len(names)), names)
plt.ylim(-1, 3);

On the x-axis, the absolute sentence number within the corpus is indicated; and each vertical bar indicates a position where the target word occurs.

> Then Shahrazad reached the morning, and fell silent in the telling of her tale…

## A class for a night

In the lecture, we bundled the preprocessing of a text and its statistics in a class. Let's do the same for the nights: every night becomes an object that knows its number, its sentences, and how long it takes to tell.

### Exercise 4

Write a class `Night` with a constructor that takes a `filepath` and

* stores the file path in an attribute `filepath`;
* stores the number of the night in an attribute `number` (use `get_night`);
* reads and preprocesses the file, and stores the result in an attribute `sentences` (a list of sentences, each a list of words; use `read_file` and `preprocess`).

Add three methods: `sentence_count()`, `word_count()` and `story_time()`, which return what their names say. Test the class on `arabian_nights/1.txt`: the first night has 11372 words.

In [ ]:
# insert your code here
class night:
  def __init__(self, filepath):
    self.filepath = filepath

    self.number = get_night(self.filepath)
    text = read_file(self.filepath)
    self.sentences = preprocess(text)

  def sentence_count(self):
      return(len(self.sentences))

  def word_count(self):
      return(count_words(self.sentences))

  def story_time(self):
      return(story_time(self.sentences))

result = night('arabian_nights/1.txt')
print(result.word_count())

## Collocations

In the next part of this practical session, our aim is to automatically come up with a list of collocations that are present in (an excerpt from) the English Wikipedia. As we've seen during the lecture, collocations are word combinations that appear together much more frequently than one would expect based on their individual frequencies (e.g. *red wine* is a collocation, because *red wine* as a combination appears much more frequently than one would expect based on the frequencies of *red* and *wine* individually).

For our analysis, we'll first make sure that our Wikipedia corpus is properly tokenized. Additionally we'll also remove stop words from our data (stop words are very frequent words such as *the*, *of* and *and*, which carry little meaning of their own).

Let's first make sure that our list of English stopwords is downloaded, so that we can make use of it in `NLTK`. We'll also make sure that `NLTK`'s stopword functionality is properly loaded by importing the proper module.

In [ ]:
nltk.download('stopwords')
from nltk.corpus import stopwords

Next, we'll read in our Wikipedia text file, and tokenize it using the `tokenize_and_filter` function we defined before. Note that we're not using the `preprocess` function, because we want all our tokens in a single list, not separated per sentence.

In [ ]:
wiki_string = read_file('wiki/wiki_sample_en_small.txt')
wiki_tokens = tokenize_and_filter(wiki_string)

### Exercise 5

Using `stopwords.words('english')`, you can now access a list of stopwords in English. Create a new variable `wiki_tokens_filtered`, which has all the stopwords filtered out. Tip: checking whether a word is in a *set* is much faster than checking whether it is in a list, so first convert the stopword list to a set.

In [ ]:
# insert your code here


Let's now make sure that `NLTK`'s collocation functionality is properly loaded. We'll make use of `BigramAssocMeasures`, which contains the actual collocation measures, and `BigramCollocationFinder`, which we'll use to construct word pairs (together with their frequencies) from our text.

In [ ]:
from nltk.collocations import BigramAssocMeasures
from nltk.collocations import BigramCollocationFinder
bigram_measures = BigramAssocMeasures()

We can now make use of the `from_words` method of the `BigramCollocationFinder` class in order to automatically construct a set of word pairs and their frequencies from the data. This data can then be explored using a number of collocation measures, all implemented in NLTK (defined in the `BigramAssocMeasures` class).

In [ ]:
finder = BigramCollocationFinder.from_words(wiki_tokens_filtered)

First we'll apply a frequency filter; this means we're only keeping candidate collocations that appear with a frequency of at least $n$ (3 in this case):

In [ ]:
finder.apply_freq_filter(3)

Next, we'll select those word pairs that have the highest collocation score (in this case, we're using the raw frequency, i.e. `raw_freq`, as a score).

In [ ]:
finder.nbest(bigram_measures.raw_freq, 20)

### Exercise 6

You might have noticed that raw frequency is not the best possible collocation measure. Inspect the `BigramAssocMeasures` class in order to learn what other measures are available. Experiment with a number of those measures and inspect the results. Which measures work best? Hint: you can always get more information for a particular module or function by typing `help(module)`, i.e. `help(BigramAssocMeasures)` in this case. You'll also find some more information about possible measures implemented in `NLTK` on the internet.

In [ ]:
# insert your code here


Note: some measures tend to overestimate infrequent collocations. Make sure to jointly experiment with the frequency threshold, and check which threshold works best for what measure.

## Spacy

Next, we'll experiment with Spacy's preprocessing pipeline. First, we'll import `spacy`. This should work out of the box in Google Colab. If you're doing this practical session on your own computer, and you haven't installed spacy before, you will need to install it (e.g. using the Anaconda Navigator).

In [ ]:
import spacy

Next, we'll load the preprocessing models for English. Again, this will work out of the box on Google Colab. If you're working locally, you will need to make sure to download the English model files as well. The error message will tell you how to do so. You can find more information on Spacy's [Models & Languages page](https://spacy.io/usage/models).

In [ ]:
nlp = spacy.load('en_core_web_sm')

Now we're ready to start preprocessing text. Let's see how well Spacy does on a random newspaper article. Copy a news article from your favourite English news website (e.g. [The Guardian](https://www.theguardian.com/international)), and paste it in the code cell below (make sure you leave the triple quotation marks, as we're defining a multi-line string here):

In [ ]:
news_article = """
COPY YOUR ARTICLE HERE
"""

We can now give our news article (defined as a string of characters) to spacy's processing pipeline, using the code below. The article, together with all preprocessing annotations, will be put in the `doc` variable.

In [ ]:
doc = nlp(news_article)

### Exercise 7

**(1)** Explore the various annotations that are provided by the preprocessing: explore the results of the tokenization, the sentence detection, and the lemmatization. As you go through the results you're likely to notice that the results contain some errors, or things that you would have annotated differently. It's important to realize that this will always be the case: the preprocessing pipeline is an automatic process, and the final results are likely to contain some errors.

In [ ]:
# insert your code here


**(2)** Every token has an attribute `pos_` with its part of speech (`NOUN`, `VERB`, `ADJ`, ...). Count how often each part of speech occurs in the article (a dictionary or a `Counter`), and print the counts from most to least frequent.

In [ ]:
# insert your code here


## Errors and exceptions

In the lecture, we saw that Python stops with an exception when something goes wrong, and that `try` and `except` let us handle an exception and continue. Let's practise on files, where things go wrong often: a file may be missing, or it may be in an encoding we did not expect. The cell below creates a small file in Latin-1 encoding, so that we have one.

In [ ]:
with open('document_latin1.txt', 'w', encoding='latin1') as out_file:
    out_file.write('Un texte français: café, déjà vu.\n')

### Exercise 8

Write a function `safe_read(filename)` that reads a file and returns its contents as a string, like `read_file` above. But if the file does not exist (`FileNotFoundError`), or if it cannot be decoded as UTF-8 (`UnicodeDecodeError`), the function prints a message that says which file caused the problem, and returns `None`. Use one `try` block with two `except` clauses.

Then loop over the list `test_files` below, and print the length of every file that could be read.

In [ ]:
test_files = ['arabian_nights/1.txt', 'arabian_nights/2000.txt', 'document_latin1.txt', 'arabian_nights/2.txt']

# insert your code here


### Exercise 9

Each of the two cells below contains a small bug. Run each cell, read the traceback from the bottom up, name the type of exception and the line that causes it, and fix the code. Write the exception type in a comment above your fix.

In [ ]:
# cell 1: the number of words of the first ten nights
word_counts = {}
for night in corpus[:10]:
    word_counts[get_night(night)] = count_words(night)
print(word_counts)

In [ ]:
# cell 2: the first word of the first sentence of the first night
first_sentence = corpus[0][0]
print(first_sentence.split()[0])

## Challenge exercises

If you have finished the exercises above and want more practice, here are a few additional exercises. They are optional.

### Challenge 1

Now write a class `Nights` that holds the whole collection (composition: a `Nights` object *has* a list of `Night` objects). Its constructor takes a glob pattern such as `'arabian_nights/*.txt'`, creates a `Night` object for every file, and stores them in an attribute `nights`, sorted by night number. Add the methods:

* `total_time()` returns the total story time in minutes;
* `longest()` returns the `Night` object with the most words.

Building all 999 nights takes a minute, since every file is tokenized again. Which night is the longest, and how many hours does the whole collection take to tell?

In [ ]:
# insert your code here


### Challenge 2

Which words are typical for a particular night? A simple measure: compare the relative frequency of a word in one night with its relative frequency in the whole corpus. Write a function `distinctive_words(night, n)` that takes a night from `corpus` and returns the `n` words with the highest ratio

$$\frac{\textrm{frequency in the night} \,/\, \textrm{words in the night}}{\textrm{frequency in the corpus} \,/\, \textrm{words in the corpus}}$$

considering only words that occur at least 5 times in the night (otherwise rare words dominate). You will need a frequency count for the whole corpus; compute it once, outside the function. Try the function on a few nights.

In [ ]:
# insert your code here
